# P01 · Formal Verification Evidence Pipeline
### 2-to-1 multiplexer — SymbiYosys · Yosys · Z3

Takes **two versions of the same mux** (one correct, one deliberately wrong), checks both against **one shared
assertion**, and produces a **sealed, hash-verified evidence package** for independent audit.

| | |
|---|---|
| **Chain** | R → I → S → RTL → **F** → C &nbsp;(Requirement → Interpretation → Specification → RTL → *Formal evidence* → Certification) |
| **Specification** | `y = sel ? b : a` |
| **Expected** | correct RTL → formal **PASS** · wrong RTL → formal **FAIL** + counterexample witness |
| **This notebook produces** | **F** — tool-generated evidence. It does **not** certify (**C**); that needs an independent audit. |

#### Pipeline

```
pinned source ─► toolchain ─► run SBY (reference + wrong) ─► collect raw artifacts
                                                                │
   sealed archive ◄─ manifest ◄─ SMT cross-check ◄─ counterexample ◄─ verify outcomes
```

#### Contents
| § | Stage | Fails closed when… |
|---|---|---|
| 1 | Configuration | — |
| 2 | Helpers | — |
| 3 | Toolchain & environment | a tool is missing |
| 4 | Source under test | HEAD ≠ pinned SHA, patch or input hashes drift |
| 5 | Execute formal runner | *(recorded, never hidden)* |
| 6 | Collect raw artifacts | — |
| 7 | Verify formal outcomes | any expected PASS/FAIL, parity or elaboration check fails |
| 8 | Counterexample inspection | witness is non-Boolean or does not violate the spec |
| 9 | Independent SMT cross-check | Z3 disagrees with SBY |
| 10 | Seal package | — |
| 11 | Verify archive & verdict | archive differs from its manifest |
| 12 | Download | — |

#### Principles
1. **Fail closed** — a missing or contradictory record is a failed check, never a silent pass.
2. **Write-once evidence** — a file is never overwritten with different content. Start over with `NEW_RUN = True`.
3. **Raw over summarised** — the package keeps native tool output; nothing is hand-written.
4. **Everything hashed** — sources, patch, artifacts, and the final archive.

**Run:** *Runtime → Run all* (Colab, ~1 min) or execute the cells top-to-bottom. The download cell is Colab-only.

## 1 · Configuration
The only cell meant for editing. The audited values (toolchain, input hashes, patch) come from the independent
audit of evidence package `p01-20260930T155000Z-rev3`.

In [ ]:
from __future__ import annotations

import datetime as dt
import gzip
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
from dataclasses import dataclass, field
from itertools import product
from pathlib import Path

NEW_RUN = False   # True → start a fresh run ID (evidence is write-once)
SEAL_REV = 1      # bump to re-seal the same run after adding material

P01_REL = "Files/Research_Papers/benchmark/pilot_v0_1/P01"
RUNNER = "formal/run_p01_formal.sh"


@dataclass(frozen=True)
class Config:
    # Source under test
    repo_url: str = "https://github.com/Kumar62812/neewan_1.git"
    pinned_sha: str = "6d09dca0e6d8292d0b4cf13ff19b3ed06ec3de35"
    apply_path_fix: bool = True          # audited .sby staged-path patch (see §4)
    # Toolchain
    sby_git_ref: str = "v0.69"
    audited_toolchain: dict = field(default_factory=lambda: {
        "yosys": "Yosys 0.33 (git sha1 2584903a060)",
        "sby": "SBY v0.69",
        "z3": "Z3 version 4.8.12 - 64 bit",
    })
    strict_toolchain: bool = False       # True → a version mismatch fails the run
    # Layout
    workdir: Path = Path(os.environ.get("P01_WORKDIR")
                         or ("/content" if Path("/content").is_dir() else Path.cwd() / "p01_work"))

    @property
    def repo_dir(self) -> Path: return self.workdir / "neewan_1"
    @property
    def p01_dir(self) -> Path: return self.repo_dir / P01_REL
    @property
    def formal_dir(self) -> Path: return self.p01_dir / "formal"
    @property
    def evidence_root(self) -> Path: return self.workdir / "p01_formal_evidence"
    @property
    def state_file(self) -> Path: return self.workdir / "p01_current_run_id.txt"


CFG = Config()
CFG.workdir.mkdir(parents=True, exist_ok=True)
if "/usr/local/bin" not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] += os.pathsep + "/usr/local/bin"

# ── Design contract (P01-specific) ───────────────────────────────────────────
def SPEC(a: int, b: int, sel: int) -> int:          # y = sel ? b : a
    return b if sel else a

def WRONG_MODEL(a: int, b: int, sel: int) -> int:   # deliberately swapped mux
    return a if sel else b

TOP = {"reference": "harness_reference", "wrong": "harness_wrong"}
FREE_INPUTS = {"a", "b", "sel"}

# ── Audited inputs: sha256 of every file under test, relative to P01/ ────────
EXPECTED_INPUT_SHA256 = {
    "formal/run_p01_formal.sh":                 "cd69b3ff122b092e5c02cc847ce3fb61b66a6368d62ae0b3de57b3e7b2ecd2b5",
    "formal/reference.sby":                     "1518c9cdcc748c8478cf6fd8a69eb815195b1cdcfda4a00c538c74b99afbda36",
    "formal/wrong.sby":                         "7ecac0479aa73888169a39d95dd378e87404257433a804da71089c0df731e4f6",
    "formal/harness_reference.sv":              "04c431dea934fb3c6fb9e82bca9a586b40877b9e824b032788ca2cc05140fb89",
    "formal/harness_wrong.sv":                  "510d90750f6926a88a060a72d6ddb474ca0d73f7b5fec6c302f271fb7cf8710d",
    "formal/properties/mux2to1_properties.sv":  "7d112c61b7c0bc3fc95c52ae8e9bf0c0b72704f5622df38fdf3ea5c2589ba45d",
    "reference_rtl/rtl_01.sv":                  "73a0d25c5807966563bdf807964084233f90fd884b041acb93e500799f8b83be",
    "validation_rtl/deliberately_wrong_rtl.sv": "8730b5491deb5cbb3c2889cb0915de6091e739f0a098c38051615c2624fde57e",
}

# ── Audited patch: SBY [script] must read the *staged* basenames in src/ ─────
PATH_FIX_PATCH_SHA256 = "71bee3408a34460b8c84bc34838779ff6cea0316662f1d6f8d35742be7e7d5c4"
PATH_FIX_PATCH = r'''diff --git a/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/reference.sby b/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/reference.sby
index 141d45a..a1163a0 100644
--- a/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/reference.sby
+++ b/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/reference.sby
@@ -7,8 +7,8 @@ expect pass
 smtbmc z3
 
 [script]
-read_verilog -formal -sv ../reference_rtl/rtl_01.sv
-read_verilog -formal -sv properties/mux2to1_properties.sv
+read_verilog -formal -sv rtl_01.sv
+read_verilog -formal -sv mux2to1_properties.sv
 read_verilog -formal -sv harness_reference.sv
 prep -top harness_reference
 
diff --git a/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/wrong.sby b/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/wrong.sby
index 3928ada..53e9ebe 100644
--- a/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/wrong.sby
+++ b/Files/Research_Papers/benchmark/pilot_v0_1/P01/formal/wrong.sby
@@ -7,8 +7,8 @@ expect fail
 smtbmc z3
 
 [script]
-read_verilog -formal -sv ../validation_rtl/deliberately_wrong_rtl.sv
-read_verilog -formal -sv properties/mux2to1_properties.sv
+read_verilog -formal -sv deliberately_wrong_rtl.sv
+read_verilog -formal -sv mux2to1_properties.sv
 read_verilog -formal -sv harness_wrong.sv
 prep -top harness_wrong
 
'''

## 2 · Helpers
Small, dependency-free building blocks: command execution, hashing, **write-once** file output, and a check registry
that every verification stage reports into.

In [ ]:
UTC = dt.timezone.utc


def utc_now() -> str:
    return dt.datetime.now(UTC).isoformat(timespec="seconds")


class PipelineError(RuntimeError):
    """The pipeline cannot continue safely."""


def sh(args, *, cwd=None, check=True) -> subprocess.CompletedProcess:
    """Run a command, capture text output, and fail loudly with context."""
    argv = [str(a) for a in args]
    proc = subprocess.run(argv, cwd=cwd, text=True, capture_output=True)
    if check and proc.returncode:
        tail = (proc.stderr or proc.stdout)[-2000:]
        raise PipelineError(f"command failed (rc={proc.returncode}): {' '.join(argv)}\n{tail}")
    return proc


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def put(path: Path, content: str | bytes) -> Path:
    """Write-once: an identical rewrite is a no-op; different content is refused."""
    data = content.encode() if isinstance(content, str) else content
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() == data:
            return path
        raise PipelineError(f"refusing to overwrite evidence: {path}\n"
                            "Evidence is write-once. Set NEW_RUN = True (§1) for a fresh run.")
    path.write_bytes(data)
    return path


def tree_hashes(root: Path) -> dict[str, str]:
    return {p.relative_to(root).as_posix(): sha256_file(p) for p in sorted(root.rglob("*")) if p.is_file()}


def copy_tree(src: Path, dst: Path) -> None:
    """Write-once directory copy (idempotent when dst already holds identical content)."""
    if dst.exists():
        if tree_hashes(dst) != tree_hashes(src):
            raise PipelineError(f"refusing to overwrite evidence directory: {dst}")
        return
    shutil.copytree(src, dst)


def read(path: Path) -> str:
    return path.read_text(errors="replace") if path.is_file() else ""


def banner(title: str) -> None:
    print(f"\n{'━' * 78}\n {title}\n{'━' * 78}")


@dataclass
class Check:
    stage: str
    name: str
    ok: bool
    detail: str = ""


CHECKS: list[Check] = []


def check(stage: str, name: str, ok: bool, detail: str = "") -> bool:
    """Record one verification result (re-running a cell replaces its earlier result)."""
    CHECKS[:] = [c for c in CHECKS if (c.stage, c.name) != (stage, name)]
    CHECKS.append(Check(stage, name, bool(ok), detail))
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"  — {detail}" if detail else ""))
    return bool(ok)


def resolve_run_id(cfg: Config, *, new: bool) -> str:
    if cfg.state_file.exists() and not new:
        run_id = cfg.state_file.read_text().strip()
    else:
        run_id = "p01-" + dt.datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
        cfg.state_file.write_text(run_id + "\n")
    if not re.fullmatch(r"p01-\d{8}T\d{6}Z", run_id):
        raise PipelineError(f"invalid run ID in {cfg.state_file}: {run_id!r}")
    return run_id


RUN_ID = resolve_run_id(CFG, new=NEW_RUN)
PKG = CFG.evidence_root / RUN_ID
for sub in ("provenance", "environment", "execution", "artifacts", "counterexample", "audit"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)

print(f"run id   : {RUN_ID}")
print(f"package  : {PKG}")

## 3 · Toolchain & environment
Installs anything missing (idempotent), then records exact versions and paths. A version that differs from the
audited toolchain is **flagged in the record**; set `strict_toolchain=True` to make it fatal.

In [ ]:
banner("§3  Toolchain & environment")


def ensure_toolchain(cfg: Config) -> None:
    if not (shutil.which("yosys") and shutil.which("z3")):
        sh(["apt-get", "update", "-qq"])
        sh(["apt-get", "install", "-y", "-qq", "yosys", "z3"])
    if not shutil.which("sby"):
        src = cfg.workdir / "sby_src"
        if not src.exists():
            sh(["git", "clone", "--quiet", "https://github.com/YosysHQ/sby.git", src])
        sh(["git", "-C", src, "checkout", "--quiet", cfg.sby_git_ref])
        sh(["make", "-C", src, "install"])


def capture_environment(cfg: Config, pkg: Path) -> dict[str, str]:
    commands = {"yosys": ["yosys", "-V"], "sby": ["sby", "--version"], "z3": ["z3", "--version"]}
    env_dir, found = pkg / "environment", {}
    for tool, cmd in commands.items():
        path = shutil.which(tool)
        if not path:
            raise PipelineError(f"required tool not found on PATH: {tool}")
        proc = sh(cmd, check=False)
        found[tool] = (proc.stdout or proc.stderr).strip().splitlines()[0]
        put(env_dir / f"{tool}-version.txt", found[tool] + "\n")
        put(env_dir / f"{tool}-path.txt", path + "\n")
    put(env_dir / "uname.txt", sh(["uname", "-a"]).stdout)
    put(env_dir / "os-release.txt", read(Path("/etc/os-release")))
    put(env_dir / "python-version.txt", sys.version + "\n")
    if not (env_dir / "capture-utc.txt").exists():
        put(env_dir / "capture-utc.txt", utc_now() + "\n")
    return found


ensure_toolchain(CFG)
TOOLS = capture_environment(CFG, PKG)
for tool, version in TOOLS.items():
    audited = CFG.audited_toolchain[tool]
    matches = version == audited
    if CFG.strict_toolchain:
        check("toolchain", f"{tool} matches audited version", matches, version)
    else:
        print(f"  {'OK  ' if matches else 'WARN'}  {tool:<6} {version}" + ("" if matches else f"   (audited: {audited})"))

## 4 · Source under test
Checks out the **pinned SHA**, applies the **audited path-fix patch** (uncommitted — the tested tree is identified by
pinned SHA + patch hash), verifies every input hash, and records full provenance.

> **Why a patch?** At the pinned SHA, `reference.sby` / `wrong.sby` pass source-tree paths to `read_verilog`, but SBY
> runs Yosys inside `<task>/src/` where only the *staged basenames* exist. The patch rewrites those four lines.

In [ ]:
banner("§4  Source under test")


def git(cfg: Config, *args, check_rc=True) -> subprocess.CompletedProcess:
    return sh(["git", "-C", cfg.repo_dir, *args], check=check_rc)


def prepare_source(cfg: Config, pkg: Path) -> None:
    prov = pkg / "provenance"
    if not (cfg.repo_dir / ".git").is_dir():
        sh(["git", "clone", "--quiet", cfg.repo_url, cfg.repo_dir])
    git(cfg, "fetch", "--quiet", "origin")
    git(cfg, "checkout", "--quiet", "--detach", "--force", cfg.pinned_sha)
    git(cfg, "reset", "--quiet", "--hard", cfg.pinned_sha)
    git(cfg, "clean", "-fdxq")

    head = git(cfg, "rev-parse", "HEAD").stdout.strip()
    check("source", "HEAD equals pinned SHA", head == cfg.pinned_sha, head[:12])

    patch_ok = hashlib.sha256(PATH_FIX_PATCH.encode()).hexdigest() == PATH_FIX_PATCH_SHA256
    check("source", "embedded patch matches audited hash", patch_ok, PATH_FIX_PATCH_SHA256[:12])
    patch_file = put(prov / "path-fix.patch", PATH_FIX_PATCH)

    if cfg.apply_path_fix:
        already = git(cfg, "apply", "--check", "--reverse", patch_file, check_rc=False).returncode == 0
        if not already:
            git(cfg, "apply", "--check", patch_file)
            git(cfg, "apply", patch_file)
        print(f"  patch {'already present upstream' if already else 'applied'}")

    # provenance
    put(prov / "pinned-sha.txt", cfg.pinned_sha + "\n")
    put(prov / "tested-tree.txt",
        f"base_sha={cfg.pinned_sha}\npatch_applied={cfg.apply_path_fix}\npatch_sha256={PATH_FIX_PATCH_SHA256}\n")
    put(prov / "repository-url.txt", cfg.repo_url + "\n")
    put(prov / "head-commit-metadata.txt", git(cfg, "show", "-s", "--format=fuller", "HEAD").stdout)
    put(prov / "diff-vs-pinned.patch", git(cfg, "diff", cfg.pinned_sha).stdout)
    put(prov / "worktree-status.txt", git(cfg, "status", "--short").stdout)

    # input hashes: recorded in `sha256sum -c` format, then compared with the audited values
    actual = {rel: sha256_file(cfg.p01_dir / rel) for rel in EXPECTED_INPUT_SHA256}
    put(prov / "input-hashes.sha256", "".join(f"{h}  {rel}\n" for rel, h in sorted(actual.items())))
    for rel, expected in EXPECTED_INPUT_SHA256.items():
        check("source", f"input hash · {rel}", actual[rel] == expected, actual[rel][:12])


prepare_source(CFG, PKG)

## 5 · Execute the formal runner
Runs the repository's own `run_p01_formal.sh` (the audited runner) and records the exact command, UTC start/end,
stdout, stderr and exit status. A non-zero exit is **recorded, not hidden** — artifacts are still collected in §6 and
§7 turns the failure into a failed check.

In [ ]:
banner("§5  Execute formal runner")


def run_runner(cfg: Config, pkg: Path) -> int:
    ex = pkg / "execution"
    cmd = ["bash", "./" + Path(RUNNER).name]
    put(ex / "command.txt", f"cd {cfg.formal_dir}\n{' '.join(cmd)}\n")
    put(ex / "git-status-before.txt", git(cfg, "status", "--short").stdout)
    put(ex / "tested-head.txt", git(cfg, "rev-parse", "HEAD").stdout)

    started = utc_now()
    proc = subprocess.run(cmd, cwd=cfg.formal_dir, text=True, capture_output=True)
    ended = utc_now()

    put(ex / "start-utc.txt", started + "\n")
    put(ex / "end-utc.txt", ended + "\n")
    put(ex / "stdout.log", proc.stdout)
    put(ex / "stderr.log", proc.stderr)
    put(ex / "exit-status.txt", f"{proc.returncode}\n")
    return proc.returncode


EXIT_CODE = run_runner(CFG, PKG)
print("\n".join((read(PKG / "execution" / "stdout.log")).splitlines()[-12:]))
if read(PKG / "execution" / "stderr.log").strip():
    print("\nstderr:\n" + read(PKG / "execution" / "stderr.log"))
print(f"\nrunner exit status: {EXIT_CODE}")

## 6 · Collect raw artifacts
Copies the native tool output — SBY task directories, logs, models, traces — into the package **unchanged**.

In [ ]:
banner("§6  Collect raw artifacts")


def collect_artifacts(cfg: Config, pkg: Path) -> None:
    art = pkg / "artifacts"
    mapping = {"reference": "reference-run", "wrong": "wrong-run", "evidence": "runner-evidence"}
    for src_name, dst_name in mapping.items():
        src = cfg.formal_dir / src_name
        if src.is_dir():
            copy_tree(src, art / dst_name)
            print(f"  preserved  {src_name:<9} → artifacts/{dst_name}")
        else:
            print(f"  absent     {src_name}")
    put(art / "git-status-after-run.txt", git(cfg, "status", "--short").stdout)
    put(art / "git-diff-after-run.txt", git(cfg, "diff").stdout)


collect_artifacts(CFG, PKG)

## 7 · Verify formal outcomes
Does not rely on a shell exit code. Every claim is re-derived from SBY's own status files, logs and models:

* **Outcomes** — reference PASS, wrong FAIL, both as SBY reports them
* **Fairness** — identical property file, identical `[options]`/`[engines]`, staged inputs match the audited hashes
* **Elaboration** — both designs compile; the assertion exists in both elaborated models
* **Witness** — the wrong run produced its trace files; the reference produced none
* **Free-input interpretation** — Yosys' *"no driver"* warnings are expected: `a`, `b`, `sel` must be free inputs
  (`anyseq`) in both SMT models

In [ ]:
banner("§7  Verify formal outcomes")


def sby_sections(text: str) -> dict[str, list[str]]:
    sections, current = {}, None
    for raw in text.splitlines():
        line = raw.strip()
        if m := re.fullmatch(r"\[(\w+)[^\]]*\]", line):
            current = m.group(1)
            sections[current] = []
        elif line and current:
            sections[current].append(line)
    return sections


def anyseq_names(smt2: str) -> set[str]:
    names = set()
    for ident in re.findall(r"yosys-smt2-anyseq (\S+) ", smt2):
        if m := re.search(rf"\(declare-fun \|{re.escape(ident)}\|.*?; \\(\w+)", smt2):
            names.add(m.group(1))
    return names


def verify_formal(pkg: Path) -> None:
    S = "formal"
    art, ex = pkg / "artifacts", pkg / "execution"
    runs = {"reference": (art / "reference-run", "PASS"), "wrong": (art / "wrong-run", "FAIL")}

    check(S, "runner exit status is 0", read(ex / "exit-status.txt").strip() == "0",
          f"exit={read(ex / 'exit-status.txt').strip() or 'missing'}")
    rs = dict(re.findall(r"(\w+)=(\d+)", read(art / "runner-evidence" / "status.txt")))
    check(S, "runner recorded reference_sby_exit=0 and wrong_sby_exit=0",
          rs.get("reference_sby_exit") == "0" and rs.get("wrong_sby_exit") == "0", str(rs))

    for name, (d, marker) in runs.items():
        log = read(d / "logfile.txt")
        status_word = (read(d / "status").split() or ["missing"])[0]
        check(S, f"{name}: SBY status file = {marker}", status_word == marker, status_word)
        check(S, f"{name}: marker file {marker} exists", (d / marker).is_file())
        check(S, f"{name}: log reports DONE ({marker}, rc=0)", f"DONE ({marker}, rc=0)" in log)
        check(S, f"{name}: elaboration and SMT2 generation succeeded",
              "prep: finished (returncode=0)" in log and "smt2: finished (returncode=0)" in log)
        check(S, f"{name}: assertion present in elaborated model",
              "$assert" in read(d / "model" / "design_prep.il"))

    ref_log, wrong_log = (read(runs[k][0] / "logfile.txt") for k in ("reference", "wrong"))
    check(S, "reference: engine reports pass", "Status returned by engine: pass" in ref_log)
    check(S, "wrong: BMC failed on the shared property file",
          "BMC failed!" in wrong_log and re.search(r"Assert failed in .*mux2to1_properties\.sv", wrong_log) is not None)

    # fairness / parity
    ref_cfg, wrong_cfg = (sby_sections(read(runs[k][0] / "config.sby")) for k in ("reference", "wrong"))
    strip = lambda opts: [o for o in opts if not o.startswith("expect")]
    check(S, "identical [options] (except expect) and [engines]",
          strip(ref_cfg.get("options", [])) == strip(wrong_cfg.get("options", []))
          and ref_cfg.get("engines") == wrong_cfg.get("engines")
          and ref_cfg.get("engines") is not None,
          f"{' / '.join(strip(ref_cfg.get('options', [])))} · {(ref_cfg.get('engines') or ['?'])[0]}")
    check(S, "expect pass (reference) / expect fail (wrong)",
          "expect pass" in ref_cfg.get("options", []) and "expect fail" in wrong_cfg.get("options", []))
    prop = "mux2to1_properties.sv"
    p_ref, p_wrong = (runs[k][0] / "src" / prop for k in ("reference", "wrong"))
    check(S, "shared property file is byte-identical in both runs",
          p_ref.is_file() and p_wrong.is_file() and sha256_file(p_ref) == sha256_file(p_wrong))
    staged_ok = all(
        (runs[k][0] / "src" / Path(rel).name).is_file()
        and sha256_file(runs[k][0] / "src" / Path(rel).name) == EXPECTED_INPUT_SHA256[rel]
        for k, rels in (("reference", ["reference_rtl/rtl_01.sv", "formal/harness_reference.sv", f"formal/properties/{prop}"]),
                        ("wrong", ["validation_rtl/deliberately_wrong_rtl.sv", "formal/harness_wrong.sv", f"formal/properties/{prop}"]))
        for rel in rels)
    check(S, "staged src/ files match audited input hashes", staged_ok)

    # witness artifacts
    wtr = runs["wrong"][0] / "engine_0"
    traces = ["trace.vcd", "trace.yw", "trace_tb.v", "trace.smtc"]
    check(S, "wrong: all four trace files generated", all((wtr / t).is_file() for t in traces))
    check(S, "reference: no counterexample trace (expected for a pass)",
          not any((runs["reference"][0] / "engine_0").glob("trace*")))

    # free-input interpretation of the "no driver" warnings
    for name, (d, _) in runs.items():
        warned = set(re.findall(r"Wire \S+?\.\\(\w+) is used but has no driver", read(d / "logfile.txt")))
        free = anyseq_names(read(d / "model" / "design_smt2.smt2"))
        check(S, f"{name}: 'no driver' wires are exactly the free inputs {sorted(FREE_INPUTS)}",
              warned == FREE_INPUTS, f"warned={sorted(warned)}")
        check(S, f"{name}: SMT2 model declares them anyseq (free) — no hidden assumption",
              free == FREE_INPUTS, f"anyseq={sorted(free)}")


verify_formal(PKG)

## 8 · Counterexample inspection
Parses the tool-generated VCD **with timestamps**, takes the signal state at the failing step, and recomputes the
required output from the specification. The top-level signals must agree with the DUT and property-module copies.

In [ ]:
banner("§8  Counterexample inspection")


def parse_vcd(text: str):
    """Return (top_module, [(time, state, changed_signal_names), …])."""
    scope, names, state, snaps, top = [], {}, {}, [], None
    t, changed = None, set()
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith("$scope"):
            scope.append(line.split()[2])
            top = top or scope[0]
        elif line.startswith("$upscope"):
            scope.pop()
        elif line.startswith("$var"):
            parts = line.split()
            names.setdefault(parts[3], []).append(".".join(scope + [parts[4]]))
        elif line.startswith("$"):
            continue
        elif line.startswith("#"):
            if t is not None:
                snaps.append((t, dict(state), frozenset(changed)))
            t, changed = int(line[1:]), set()
        else:
            value, ident = line[1:].split() if line[0] in "bB" else (line[0], line[1:])
            for name in names.get(ident, []):
                state[name] = value
                changed.add(name)
    if t is not None:
        snaps.append((t, dict(state), frozenset(changed)))
    return top, snaps


def inspect_counterexample(pkg: Path) -> None:
    S = "counterexample"
    cex = pkg / "counterexample"
    src = pkg / "artifacts" / "wrong-run" / "engine_0"
    if not (src / "trace.vcd").is_file():
        check(S, "witness VCD available", False, "wrong-run trace missing")
        return
    for name in ("trace.vcd", "trace.yw"):
        put(cex / name, (src / name).read_bytes())

    top, snaps = parse_vcd(read(src / "trace.vcd"))
    design = [s for s in snaps if any(not n.startswith("smt_") for n in s[2])]
    t, state, _ = design[-1]                                   # the failing step
    sig = lambda name, scope="": state.get(f"{top}.{scope}{name}")
    a, b, sel, y = (sig(n) for n in ("a", "b", "sel", "y"))

    copies = {scope: tuple(sig(n, scope) for n in ("a", "b", "sel", "y")) for scope in ("dut.", "props.")}
    check(S, "top-level, dut and props copies of a/b/sel/y agree",
          all(v == (a, b, sel, y) for v in copies.values()), str(copies))
    boolean = all(v in {"0", "1"} for v in (a, b, sel, y))
    check(S, "witness values are legal Booleans", boolean, f"a={a} b={b} sel={sel} y={y}")
    if not boolean:
        return
    required = SPEC(int(a), int(b), int(sel))
    mismatch = required != int(y)
    check(S, "observed y differs from the specification", mismatch,
          f"a={a} b={b} sel={sel} → required y={required}, observed y={y}")

    report = {"vcd_top": top, "failing_step_time": t, "inputs": {"a": a, "b": b, "sel": sel},
              "required_y": required, "observed_y": int(y), "mismatch": mismatch}
    put(cex / "inspection.json", json.dumps(report, indent=2) + "\n")
    put(cex / "inspection-output.txt",
        f"Counterexample (VCD time {t}, top {top})\n"
        f"  inputs      : a={a}  b={b}  sel={sel}\n"
        f"  spec        : y = sel ? b : a  →  required y = {required}\n"
        f"  observed    : y = {y}\n"
        f"  mismatch    : {mismatch}\n")
    print("\n" + read(cex / "inspection-output.txt"))


inspect_counterexample(PKG)

## 9 · Independent SMT cross-check
A second opinion that does **not** go through SymbiYosys' verdict: load each retained `design_smt2.smt2` straight
into Z3 and enumerate all 8 `(a, b, sel)` combinations.

* reference → **no** violating combination
* wrong → violating exactly where `SPEC ≠ WRONG_MODEL` (here: every case with `a ≠ b`)

> `z3-solver` is installed with `--target` so its bundled `z3` binary can never shadow the toolchain `z3` that SBY uses.

In [ ]:
banner("§9  Independent SMT cross-check")


def load_z3(cfg: Config):
    try:
        import z3
    except ImportError:
        target = cfg.workdir / "pyz3"
        sh([sys.executable, "-m", "pip", "install", "--quiet", "--target", target, "z3-solver"])
        sys.path.insert(0, str(target))
        import z3
    return z3


def smt_crosscheck(cfg: Config, pkg: Path) -> None:
    S = "smt-crosscheck"
    z3 = load_z3(cfg)
    combos = list(product((0, 1), repeat=3))
    expected = {"reference": set(), "wrong": {c for c in combos if SPEC(*c) != WRONG_MODEL(*c)}}
    results = {}
    for name, top in TOP.items():
        model = pkg / "artifacts" / f"{name}-run" / "model" / "design_smt2.smt2"
        if not model.is_file():
            check(S, f"{name}: SMT2 model available", False)
            continue
        text, violating = read(model), set()
        for a, b, sel in combos:
            lit = lambda v: "true" if v else "false"
            solver = z3.Solver()
            solver.from_string(text + f"""
(declare-const st |{top}_s|)
(assert (|{top}_u| st)) (assert (|{top}_i| st)) (assert (|{top}_h| st))
(assert (= (|{top}_n a| st) {lit(a)}))
(assert (= (|{top}_n b| st) {lit(b)}))
(assert (= (|{top}_n sel| st) {lit(sel)}))
(assert (not (|{top}_a| st)))
""")
            verdict = solver.check()
            if verdict == z3.unknown:
                raise PipelineError(f"Z3 returned unknown for {name} {(a, b, sel)}")
            if verdict == z3.sat:
                violating.add((a, b, sel))
        results[name] = sorted(violating)
        check(S, f"{name}: Z3 violating (a,b,sel) set equals the expected set",
              violating == expected[name], f"{len(violating)}/8 violate: {sorted(violating)}")
    put(pkg / "counterexample" / "smt-crosscheck.json", json.dumps(
        {"method": "Z3 over retained design_smt2.smt2, exhaustive over (a,b,sel)",
         "z3_python": z3.get_version_string(),
         "violating": {k: [list(c) for c in v] for k, v in results.items()}}, indent=2) + "\n")


smt_crosscheck(CFG, PKG)

## 10 · Seal the package
Writes the run report, then a path-relative manifest (`sha256sum -c` compatible) and a **byte-reproducible** archive
(sorted entries, zeroed timestamps and owners). A package with any failed check is sealed with a `-FAILED` suffix so
it can never be mistaken for a clean run.

In [ ]:
banner("§10  Seal package")


def build_manifest(pkg: Path) -> dict[str, str]:
    """Hash every file except audit/ (which holds the manifests themselves)."""
    return {rel: h for rel, h in tree_hashes(pkg).items() if not rel.startswith("audit/")}


def write_archive(pkg: Path, out: Path) -> None:
    def normalise(info: tarfile.TarInfo) -> tarfile.TarInfo:
        info.uid = info.gid = 0
        info.uname = info.gname = ""
        info.mtime = 0
        info.mode = 0o755 if info.isdir() else 0o644
        return info

    with out.open("xb") as raw, gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0, compresslevel=9) as gz, \
         tarfile.open(fileobj=gz, mode="w", format=tarfile.PAX_FORMAT) as tar:
        tar.add(pkg, arcname=pkg.name, recursive=False, filter=normalise)
        for path in sorted(pkg.rglob("*")):
            tar.add(path, arcname=f"{pkg.name}/{path.relative_to(pkg).as_posix()}", recursive=False, filter=normalise)


def seal(cfg: Config, pkg: Path, rev: int) -> Path:
    ok = all(c.ok for c in CHECKS)
    report = {
        "run_id": pkg.name, "sealed_rev": rev, "verdict": "PASS" if ok else "FAIL",
        "pinned_sha": cfg.pinned_sha, "patch_sha256": PATH_FIX_PATCH_SHA256,
        "toolchain": TOOLS, "audited_toolchain": cfg.audited_toolchain,
        "runner_exit_status": EXIT_CODE,
        "checks": [c.__dict__ for c in CHECKS],
        "note": "Evidence for independent audit. F remains PENDING until the audit accepts; C is not implied.",
    }
    put(pkg / f"run-report-rev{rev}.json", json.dumps(report, indent=2) + "\n")
    manifest = build_manifest(pkg)
    put(pkg / "audit" / f"MANIFEST-rev{rev}.sha256", "".join(f"{h}  {rel}\n" for rel, h in sorted(manifest.items())))

    suffix = "" if ok else "-FAILED"
    archive = cfg.evidence_root / f"{pkg.name}-rev{rev}{suffix}.tar.gz"
    if archive.exists():
        raise PipelineError(f"archive already exists: {archive} — bump SEAL_REV to re-seal")
    write_archive(pkg, archive)
    put(archive.with_name(archive.name + ".sha256"), f"{sha256_file(archive)}  {archive.name}\n")
    print(f"  manifest entries : {len(manifest)}")
    print(f"  archive          : {archive.name}")
    return archive


ARCHIVE = seal(CFG, PKG, SEAL_REV)

## 11 · Verify archive & verdict
Re-opens the **sealed archive** (not the working tree), recomputes every member hash against the manifest, and prints
the verdict. The final cell raises if anything failed, so the notebook status is honest in CI and `papermill` too.

In [ ]:
banner("§11  Verify archive & verdict")


def verify_archive(archive: Path, rev: int) -> str:
    S, prefix = "archive", archive.name.split(f"-rev{rev}")[0]
    manifest_name = f"{prefix}/audit/MANIFEST-rev{rev}.sha256"
    members: dict[str, str] = {}
    manifest_text = ""
    with tarfile.open(archive, "r:gz") as tar:
        for member in tar.getmembers():
            if member.isfile():
                data = tar.extractfile(member).read()
                members[member.name] = sha256_bytes(data)
                if member.name == manifest_name:
                    manifest_text = data.decode()
    manifest = {}
    for line in manifest_text.splitlines():
        digest, rel = line.split("  ", 1)
        manifest[f"{prefix}/{rel}"] = digest
    check(S, "manifest found inside the archive", bool(manifest), manifest_name)
    bad = [n for n, h in manifest.items() if members.get(n) != h]
    extra = [n for n in members if n not in manifest and "/audit/" not in n]
    check(S, "every manifest entry matches the archived bytes", bool(manifest) and not bad,
          f"{len(manifest)} files" if not bad else str(bad[:3]))
    check(S, "no unlisted files outside audit/", not extra, str(extra[:3]))
    digest = sha256_file(archive)
    recorded = read(archive.with_name(archive.name + ".sha256")).split()[0]
    check(S, "archive SHA-256 equals its sidecar file", digest == recorded)
    return digest


ARCHIVE_SHA256 = verify_archive(ARCHIVE, SEAL_REV)

by_stage: dict[str, list[Check]] = {}
for c in CHECKS:
    by_stage.setdefault(c.stage, []).append(c)
print(f"\n{'stage':<16}{'passed':>8}{'failed':>8}")
for stage, items in by_stage.items():
    print(f"{stage:<16}{sum(c.ok for c in items):>8}{sum(not c.ok for c in items):>8}")

failed = [c for c in CHECKS if not c.ok]
print(f"\narchive : {ARCHIVE.name}")
print(f"sha256  : {ARCHIVE_SHA256}")
if failed:
    print("\nVERDICT : FAIL — sealed as -FAILED; fix the causes below and start a NEW_RUN.")
    for c in failed:
        print(f"   ✘ [{c.stage}] {c.name} {c.detail}")
    raise PipelineError(f"{len(failed)} check(s) failed")
print("\nVERDICT : EVIDENCE COMPLETE — ready for independent audit.")
print("          F stays PENDING until the audit accepts. This notebook does not certify (C).")

## 12 · Download
Colab only. Keep **both** files together — the `.sha256` lets anyone recheck the archive later with
`sha256sum -c`.

In [ ]:
def offer_download(*paths: Path) -> None:
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab — files are at:")
        for p in paths:
            print(f"  {p}")
        return
    for p in paths:
        files.download(str(p))


offer_download(ARCHIVE, ARCHIVE.with_name(ARCHIVE.name + ".sha256"))